# FinanceBench — Full Pipeline
**Project:** Evaluating the Reliability of Selected RAG Evaluation Metrics for Finance-Related QA
**Student:** Ayusha Shrestha

**Dataset:** FinanceBench (Islam et al., 2023) — 150 expert-annotated questions on real SEC
10-K/10-Q/8-K filings from 40+ public companies.

**Structure confirmed from Hugging Face:** each row has a `question`, an expert-written
`answer`, and pre-supplied `evidence` (a list of evidence passages) — similar to `finqa`
in that no retrieval step is needed (evidence is already given per question), and no
separate reference answer exists beyond the `answer` itself (same situation as `finqa`,
different from MultiHop-RAG which had a distinct short gold answer).

**Methodological choices, consistent with earlier datasets:**
- Treat `answer` as the response being evaluated (same approach as `finqa` — no
  generation step needed, since evidence is pre-supplied per question)
- Use the **reference-free** Context Precision variant (same as `finqa`, for the same
  reason: no independent reference distinct from the response)
- Use FinanceBench's own native **`question_reasoning`** label (e.g. "Numerical
  reasoning", "Information extraction", "Logical reasoning") instead of a heuristic
  tagger — the third dataset in a row using dataset-provided ground-truth labels rather
  than guesswork, consistent with the MultiHop-RAG approach
- Rows with `question_type == "novel-generated"` have no `question_reasoning` label and
  are set aside from the main comparison, for the same reason MultiHop-RAG's `null_query`
  rows were set aside — a real gap in the label, not something to paper over


## 1. Install a compatible, pinned set of libraries
Same pinned stack used throughout this project.

In [ ]:
!pip install -q \
  "ragas==0.2.15" \
  "langchain==0.3.27" "langchain-core==0.3.76" "langchain-community==0.3.30" \
  "langchain-openai==0.2.14" "langchain-groq==0.2.4" "langchain-huggingface==0.1.2" \
  datasets sentence-transformers

print("Libraries installed.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.9/190.9 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the sou

## 2. Add your API keys

In [ ]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Paste your Groq API key here: ")
print("Groq key stored for this session.")


Paste your Groq API key here: ··········
Groq key stored for this session.


In [ ]:
from huggingface_hub import login

login(getpass("Paste your Hugging Face token here: "))


Paste your Hugging Face token here: ··········


## 3. Load FinanceBench and inspect it

Quick confirmation the structure matches what we expect before building on top of it.


In [ ]:
from datasets import load_dataset
import pandas as pd

fb_dataset = load_dataset("PatronusAI/financebench")
df_fb = fb_dataset["train"].to_pandas()

print("Shape:", df_fb.shape)
print("Columns:", list(df_fb.columns))
print()
print("question_type value counts:")
print(df_fb["question_type"].value_counts())
print()
print("question_reasoning value counts (including missing):")
print(df_fb["question_reasoning"].value_counts(dropna=False))


README.md:   0%|          | 0.00/1.41k [00:00<?, ?B/s]

financebench_merged.jsonl:   0%|          | 0.00/958k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/150 [00:00<?, ? examples/s]

Shape: (150, 15)
Columns: ['financebench_id', 'company', 'doc_name', 'question_type', 'question_reasoning', 'domain_question_num', 'question', 'answer', 'justification', 'dataset_subset_label', 'evidence', 'gics_sector', 'doc_type', 'doc_period', 'doc_link']

question_type value counts:
question_type
metrics-generated    50
domain-relevant      50
novel-generated      50
Name: count, dtype: int64

question_reasoning value counts (including missing):
question_reasoning
None                                                                                            50
Numerical reasoning                                                                             43
Information extraction                                                                          31
Numerical reasoning OR Logical reasoning                                                         6
Logical reasoning (based on numerical reasoning)                                                 5
Logical reasoning (based on nume

## 4. Bucket the native reasoning labels and filter out unlabeled rows


In [ ]:
def bucket_reasoning(value):
    if pd.isna(value):
        return None
    v = value.lower()
    if "numerical" in v:
        return "Numerical reasoning"
    elif "information extraction" in v:
        return "Information extraction"
    elif "logical" in v:
        return "Logical reasoning"
    else:
        return "Other"

df_fb["reasoning_bucket"] = df_fb["question_reasoning"].apply(bucket_reasoning)

df_labeled = df_fb[df_fb["reasoning_bucket"].notna()].reset_index(drop=True)
df_unlabeled = df_fb[df_fb["reasoning_bucket"].isna()].reset_index(drop=True)

print("Labeled rows (usable for main analysis):", len(df_labeled))
print(df_labeled["reasoning_bucket"].value_counts())
print()
print("Unlabeled rows (novel-generated, set aside):", len(df_unlabeled))


Labeled rows (usable for main analysis): 100
reasoning_bucket
Numerical reasoning       67
Information extraction    33
Name: count, dtype: int64

Unlabeled rows (novel-generated, set aside): 50


## 5. Stratified sample: 5 rows per reasoning bucket (n=15)

Same sampling approach as MultiHop-RAG — a manual loop and `pd.concat`, not
`groupby().apply()`, since that silently dropped the grouping column when tested earlier
in this project.


In [ ]:
# Note: only two reasoning buckets actually appear in this dataset once bucketed
# (Numerical reasoning and Information extraction) -- no row buckets purely into
# "Logical reasoning" or "Other" under these rules, since logical-reasoning labels
# in this dataset are consistently paired with numerical reasoning. This is a real
# property of FinanceBench's labels, confirmed after inspection, not an error.
# Here, I have draw more rows per bucket (8 each) to keep the total sample close to the n=15
# used for finqa and MultiHop-RAG, rather than force a third, empty category.
ROWS_PER_BUCKET = 8
samples = []
for bucket in df_labeled["reasoning_bucket"].unique():
    subset = df_labeled[df_labeled["reasoning_bucket"] == bucket]
    n = min(ROWS_PER_BUCKET, len(subset))
    samples.append(subset.sample(n=n, random_state=42))

df_sample = pd.concat(samples).reset_index(drop=True)

print("Sample shape:", df_sample.shape)
print(df_sample["reasoning_bucket"].value_counts())


Sample shape: (16, 16)
reasoning_bucket
Information extraction    8
Numerical reasoning       8
Name: count, dtype: int64


## 6. Build the RAGAS input rows

`evidence` is a list of dicts per row; I've extracted just the `evidence_text` field from
each to build the retrieved-context list.


In [ ]:
def extract_evidence_texts(evidence_list):
    return [e["evidence_text"] for e in evidence_list]

df_sample["retrieved_contexts"] = df_sample["evidence"].apply(extract_evidence_texts)

# Quick sanity check
print("Example question:", df_sample.iloc[0]["question"])
print()
print("Example answer (being evaluated as the response):", df_sample.iloc[0]["answer"])
print()
print("Number of evidence passages:", len(df_sample.iloc[0]["retrieved_contexts"]))


Example question: Which debt securities are registered to trade on a national securities exchange under Ulta Beauty's name as of FY2023?

Example answer (being evaluated as the response): There are none

Number of evidence passages: 1


## 7. Set up the LLM judge and embedding model

Same setup as `finqa` and MultiHop-RAG — Groq (Llama 3.1) as judge, local embeddings.

In [ ]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

judge_llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
ragas_llm = LangchainLLMWrapper(judge_llm)

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
ragas_embeddings = LangchainEmbeddingsWrapper(embedding_model)

print("LLM judge and embeddings ready.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

LLM judge and embeddings ready.


## 8. Run the three RAGAS metrics

Same rate-limit-safe settings used throughout: 2 requests at a time, generous timeout
and retries. Expect roughly 15-20 minutes for 15 rows, same as `finqa`.


In [ ]:
from ragas import evaluate, EvaluationDataset
from ragas.metrics import faithfulness, answer_relevancy, LLMContextPrecisionWithoutReference
from ragas.run_config import RunConfig

context_precision_metric = LLMContextPrecisionWithoutReference(llm=ragas_llm)

ragas_rows = []
for _, row in df_sample.iterrows():
    ragas_rows.append({
        "user_input": row["question"],
        "response": row["answer"],
        "retrieved_contexts": row["retrieved_contexts"],
    })

eval_dataset = EvaluationDataset.from_list(ragas_rows)

slow_and_steady = RunConfig(
    timeout=300,
    max_workers=2,
    max_retries=15,
    max_wait=90,
)

results = evaluate(
    dataset=eval_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision_metric],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
    run_config=slow_and_steady,
)

results_df = results.to_pandas()
results_df


Evaluating:   0%|          | 0/48 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[5]: TimeoutError()


,user_input,retrieved_contexts,response,faithfulness,answer_relevancy,llm_context_precision_without_reference
0,Which debt securities are registered to trade ...,[UNITED STATES\nSECURITIES AND EXCHANGE COMMIS...,There are none,1.000000,0.154718,1.0
1,"Using the cash flow statement, answer the foll...","[SQUARE, INC.\nCONSOLIDATED STATEMENTS OF CASH...",$382.00,1.000000,0.244209,NaN
2,According to the details clearly outlined with...,"[Table of Contents\nNIKE, INC.\nCONSOLIDATED B...",$16525.00,1.000000,0.321509,1.0
3,Has Boeing reported any materially important o...,[Multiple legal actions have been filed agains...,Yes. Multiple lawsuits have been filed against...,0.750000,0.625748,0.0
4,What are the major products and services that ...,[Overview\nWe are a global semiconductor compa...,AMD sells server microprocessors (CPUs) and gr...,1.000000,0.785118,1.0
5,What drove revenue change as of the FY22 for AMD?,"[Net\nrevenue for 2022 was $23.6 billion, an i...","In 2022, AMD reported Higher sales of their EP...",1.000000,0.673197,1.0
6,Using only the information within the balance ...,[Table of Contents\nCOSTCO WHOLESALE CORPORATI...,$59268.00,1.000000,0.353858,1.0
7,Has CVS Health paid dividends to common shareh...,"[Dividends\nDuring 2022, 2021 and 2020, the qu...","Yes, CVS paid a $ 0.55 dividend per share ever...",0.000000,0.787518,1.0
8,How much has the effective tax rate of Corning...,[RESULTS OF OPERATIONS\n \nThe following table...,The effective tax rate of Corning has changed ...,1.000000,0.963943,1.0
9,Does AMD have a reasonably healthy liquidity p...,"[Consolidated Balance Sheets\n \nDecember 31,\...","Yes. The quick ratio is 1.57, calculated as (c...",0.750000,0.278723,1.0


## 9. Check for and retry any failed rows

Same approach as before — free-tier timeouts are expected, not a broken pipeline.

In [ ]:
metric_cols = ["faithfulness", "answer_relevancy", "llm_context_precision_without_reference"]

missing_counts = results_df[metric_cols].isna().sum()
print("Missing values per metric:")
print(missing_counts)


Missing values per metric:
faithfulness                               0
answer_relevancy                           0
llm_context_precision_without_reference    1
dtype: int64


In [ ]:
failed_mask = results_df[metric_cols].isna().any(axis=1)
failed_indices = results_df[failed_mask].index.tolist()

print(f"Retrying {len(failed_indices)} row(s): {failed_indices}")

if len(failed_indices) > 0:
    retry_rows = [ragas_rows[i] for i in failed_indices]
    retry_dataset = EvaluationDataset.from_list(retry_rows)

    retry_results = evaluate(
        dataset=retry_dataset,
        metrics=[faithfulness, answer_relevancy, context_precision_metric],
        llm=ragas_llm,
        embeddings=ragas_embeddings,
        run_config=slow_and_steady,
    )
    retry_df = retry_results.to_pandas()

    for pos, orig_idx in enumerate(failed_indices):
        for col in metric_cols:
            if pd.isna(results_df.loc[orig_idx, col]):
                results_df.loc[orig_idx, col] = retry_df.loc[pos, col]

    print("Retry complete. Remaining missing values:")
    print(results_df[metric_cols].isna().sum())
else:
    print("No missing values - nothing to retry.")


Retrying 1 row(s): [1]


Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

Retry complete. Remaining missing values:
faithfulness                               0
answer_relevancy                           0
llm_context_precision_without_reference    0
dtype: int64


## 10. Build the final results table and save it

In [ ]:
comparison_df = pd.concat([
    df_sample[["financebench_id", "question", "reasoning_bucket", "question_type",
               "answer", "gics_sector", "doc_type"]].reset_index(drop=True),
    results_df[metric_cols].reset_index(drop=True)
], axis=1)

comparison_df.to_csv("financebench_final_results.csv", index=False)
print("Saved: financebench_final_results.csv")
comparison_df


Saved: financebench_final_results.csv


,financebench_id,question,reasoning_bucket,question_type,answer,gics_sector,doc_type,faithfulness,answer_relevancy,llm_context_precision_without_reference
0,financebench_id_00746,Which debt securities are registered to trade ...,Information extraction,domain-relevant,There are none,Consumer Discretionary,10k,1.000000,0.154718,1.0
1,financebench_id_07661,"Using the cash flow statement, answer the foll...",Information extraction,metrics-generated,$382.00,Information Technology,10k,1.000000,0.244209,1.0
2,financebench_id_03531,According to the details clearly outlined with...,Information extraction,metrics-generated,$16525.00,Consumer Discretionary,10k,1.000000,0.321509,1.0
3,financebench_id_01091,Has Boeing reported any materially important o...,Information extraction,domain-relevant,Yes. Multiple lawsuits have been filed against...,Industrials,10k,0.750000,0.625748,0.0
4,financebench_id_00995,What are the major products and services that ...,Information extraction,domain-relevant,AMD sells server microprocessors (CPUs) and gr...,Information Technology,10k,1.000000,0.785118,1.0
5,financebench_id_01198,What drove revenue change as of the FY22 for AMD?,Information extraction,domain-relevant,"In 2022, AMD reported Higher sales of their EP...",Information Technology,10k,1.000000,0.673197,1.0
6,financebench_id_04209,Using only the information within the balance ...,Information extraction,metrics-generated,$59268.00,Consumer Staples,10k,1.000000,0.353858,1.0
7,financebench_id_01244,Has CVS Health paid dividends to common shareh...,Information extraction,domain-relevant,"Yes, CVS paid a $ 0.55 dividend per share ever...",Health Care,10k,0.000000,0.787518,1.0
8,financebench_id_01346,How much has the effective tax rate of Corning...,Numerical reasoning,domain-relevant,The effective tax rate of Corning has changed ...,Information Technology,10k,1.000000,0.963943,1.0
9,financebench_id_00222,Does AMD have a reasonably healthy liquidity p...,Numerical reasoning,domain-relevant,"Yes. The quick ratio is 1.57, calculated as (c...",Information Technology,10k,0.750000,0.278723,1.0


## 11. Compare average scores by reasoning type

The main test: does the calculation/numerical-reasoning pattern seen in `finqa` and MultiHop-RAG hold up on a third, independent dataset?

In [ ]:
print("Average scores by reasoning bucket:")
print(comparison_df.groupby("reasoning_bucket")[metric_cols].mean().round(3))
print()
print("Row counts per group:")
print(comparison_df["reasoning_bucket"].value_counts())


Average scores by reasoning bucket:
                        faithfulness  answer_relevancy  \
reasoning_bucket                                         
Information extraction         0.844             0.493   
Numerical reasoning            0.792             0.547   

                        llm_context_precision_without_reference  
reasoning_bucket                                                 
Information extraction                                    0.875  
Numerical reasoning                                       0.937  

Row counts per group:
reasoning_bucket
Information extraction    8
Numerical reasoning       8
Name: count, dtype: int64


## Summary across all three datasets so far

- **`finqa`**: Calculation questions scored lower than lookup questions on Faithfulness
  and Context Precision (n=15, Llama 3.1 judge, reference-free Context Precision)
- **MultiHop-RAG**: Inference queries scored lowest on Faithfulness and Answer Relevancy;
  a wrong "No" answer scored a perfect Faithfulness/Context Precision, exposing that these
  metrics measure groundedness, not correctness
- **FinanceBench**: compare the reasoning-bucket table above — does "Numerical reasoning"
  show the same weakness pattern as `finqa`'s "calculation" category? A consistent result
  across three independently-built datasets would be strong evidence for your central
  argument; an inconsistent one is equally worth reporting honestly.

**Next: bring all three datasets' results together for the final cross-dataset comparison
and write-up.**
